# 11 · services gap

**What this notebook does.** Adds three more `source` layers -- `school`, `medical`, `emergency` -- to the village-level distance picture that `06_village_gap.ipynb` already builds for towers, NBN, MBSP, the coverage guide, and RICT. For each of the 792 BushTel villages: straight-line (haversine) distance to the nearest site of each type, plus a count of sites within `BUFFER_KM` (10 km, the same constant nb 06 uses).

It then brings **every dimension together** in one set of layered gap flags (section 4): one yes/no flag each for mobile, fixed broadband, public access, school, medical and emergency, a `gap_score` counting them, and `no_connectivity` for villages with no connectivity path at all. The flags stay separate so the report can say *why* a village is a gap, not only *that* it is.

**What it reads:**

| dataframe | file |
|---|---|
| `village_gap_df` | `data/processed/village_gap.csv` (nb 06 output, 792 rows) |
| `services_df` | `data/processed/services_sites.csv` (school/medical/emergency, built by nb 07–09) |

**Where `services_sites.csv` came from:** `medical` and `emergency` are Geoscience Australia / NHSD facility extracts with their own real coordinates (`medical_facilities_NT.csv`, `emergency_facilities_NORTHERN_TERRITORY.csv`). `school` is the NT Dept of Education public school list, which ships with **no coordinates at all** -- nb 08 gives each school its site coordinate from ACARA *School Location 2025* (stand-alone preschools take their same-named school's point), and only falls back to a locality point where ACARA has no match. See `geocode_precision` in `services_sites.csv`. Any school without a coordinate is excluded from the distance calcs below rather than guessed.

**Distances here are haversine (great-circle) on WGS84** -- consistent with how this project's distance work started (`PROJECT_CONTEXT.md` §4: "haversine now; move to EPSG:3577"). nb 06 has since moved the tower/MBSP/guide/RICT distance calcs to a projected CRS (`EPSG:3577`, via `sjoin_nearest`) for better accuracy at NT's latitude/scale; this notebook doesn't, so treat `km_nearest_*` here as very slightly less precise than the equivalent tower/MBSP columns -- still a triage number, not a coverage claim, same as everywhere else in `village_gap.columns.md`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Find the repo root from wherever Jupyter was started.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
VG_PATH = ROOT / "data" / "processed" / "village_gap.csv"
SERVICES_PATH = ROOT / "data" / "processed" / "services_sites.csv"
OUT_DIR = ROOT / "data" / "processed"; OUT_DIR.mkdir(parents=True, exist_ok=True)

BUFFER_KM = 10  # same constant nb 06 uses for the *_10km neighbourhood counts
SERVICE_GAP_KM = 15  # school / medical / emergency gap cut-off; triage, same 15 km as nb 06's GAP_KM for towers

## 1. Load the two inputs

In [2]:
village_gap_df = pd.read_csv(VG_PATH, dtype={"sa1_code": str})
assert village_gap_df["community_id"].is_unique and len(village_gap_df) == 792
print(f"village_gap.csv: {len(village_gap_df)} rows, community_id unique")

services_df = pd.read_csv(SERVICES_PATH)
services_df = services_df.dropna(subset=["latitude", "longitude"]).copy()
print(f"services_sites.csv: {len(services_df)} geocoded rows "
      f"({dict(services_df['source'].value_counts())})")

village_gap.csv: 792 rows, community_id unique
services_sites.csv: 605 geocoded rows ({'school': np.int64(273), 'medical': np.int64(181), 'emergency': np.int64(151)})


## 2. Nearest site + 10km buffer count, per source

A haversine argmin loop, run once per source (`school` / `medical` / `emergency`) against every village. (nb 06 does the equivalent for towers and MBSP with `sjoin_nearest` in EPSG:3577.)

In [3]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

BUFFER_COL = {"school": "n_schools_10km", "medical": "n_medical_10km", "emergency": "n_emergency_10km"}
village_lat = village_gap_df["latitude"].to_numpy()
village_lon = village_gap_df["longitude"].to_numpy()

results = {}
for source, prefix in [("school", "school"), ("medical", "medical"), ("emergency", "emergency")]:
    sub = services_df[services_df["source"] == source].reset_index(drop=True)
    s_lat, s_lon = sub["latitude"].to_numpy(), sub["longitude"].to_numpy()

    nearest_km = np.empty(len(village_gap_df))
    nearest_name = np.empty(len(village_gap_df), dtype=object)
    nearest_type = np.empty(len(village_gap_df), dtype=object)
    n_within_buffer = np.empty(len(village_gap_df), dtype=int)

    for i in range(len(village_gap_df)):
        d = haversine_km(village_lat[i], village_lon[i], s_lat, s_lon)
        j = np.argmin(d)
        nearest_km[i] = round(float(d[j]), 2)
        nearest_name[i] = sub["name"].iloc[j]
        nearest_type[i] = sub["type"].iloc[j] if "type" in sub.columns else None
        n_within_buffer[i] = int((d <= BUFFER_KM).sum())

    results[f"km_nearest_{prefix}"] = nearest_km
    results[f"nearest_{prefix}_name"] = nearest_name
    results[f"nearest_{prefix}_type"] = nearest_type
    results[BUFFER_COL[source]] = n_within_buffer

    print(f"{source}: n={len(sub)}  km_nearest_{prefix} median={np.median(nearest_km):.2f} "
          f"p90={np.percentile(nearest_km, 90):.2f} max={nearest_km.max():.2f}  "
          f"villages with 0 within {BUFFER_KM}km: {int((n_within_buffer==0).sum())}")

school: n=273  km_nearest_school median=22.19 p90=77.03 max=244.76  villages with 0 within 10km: 545
medical: n=181  km_nearest_medical median=30.18 p90=100.38 max=286.94  villages with 0 within 10km: 588
emergency: n=151  km_nearest_emergency median=30.93 p90=88.43 max=244.89  villages with 0 within 10km: 600


### Validate

Every village must get a value for every source (services files have no missing coordinates left after the `dropna` above), and every village coordinate must stay inside the NT bounding box nb 05 already asserts on.

In [4]:
village_gap_services_df = village_gap_df.copy()
for col, arr in results.items():
    village_gap_services_df[col] = arr

assert village_gap_services_df["km_nearest_school"].notna().all()
assert village_gap_services_df["km_nearest_medical"].notna().all()
assert village_gap_services_df["km_nearest_emergency"].notna().all()
assert village_gap_services_df["latitude"].between(-26.5, -10.0).all()
assert village_gap_services_df["longitude"].between(128.5, 138.5).all()
print("asserts passed: no missing nearest-service values; village coords in NT box")

asserts passed: no missing nearest-service values; village coords in NT box


## 3. Does adding services change the picture nb 06 already drew?

`distance_gap_candidate` (nb 06) is a **mobile-tower** gap flag. This section is a quick sanity read of it against the new services columns; the combined, every-dimension view is section 4.

In [5]:
gap = village_gap_services_df[village_gap_services_df["distance_gap_candidate"] == True]
print(f"{len(gap)} villages are distance_gap_candidate (mobile tower gap, from nb06)")
print(f"  of those, median km_nearest_school = {gap['km_nearest_school'].median():.1f}, "
      f"median km_nearest_medical = {gap['km_nearest_medical'].median():.1f}, "
      f"median km_nearest_emergency = {gap['km_nearest_emergency'].median():.1f}")

all_three_close = ((village_gap_services_df["n_schools_10km"] > 0) &
                    (village_gap_services_df["n_medical_10km"] > 0) &
                    (village_gap_services_df["n_emergency_10km"] > 0))
print(f"  villages with a school AND medical AND emergency site all within 10km: "
      f"{int(all_three_close.sum())} / 792")

384 villages are distance_gap_candidate (mobile tower gap, from nb06)
  of those, median km_nearest_school = 38.2, median km_nearest_medical = 46.2, median km_nearest_emergency = 47.7
  villages with a school AND medical AND emergency site all within 10km: 163 / 792


## 4. Layered gap flags -- every dimension, side by side

One boolean per dimension, then two summaries. Every cut-off is a constant (`SERVICE_GAP_KM` above;
`GAP_KM` = 15 km and `RICT_MATCH_KM` = 2 km live in nb 06). All distances are straight-line triage.

| column | True when | from |
|---|---|---|
| `mobile_gap` | the coverage guide claims no macro/small cell **and** the nearest carrier tower is > 15 km (= nb 06 `distance_gap_candidate`) | nb 06 |
| `fixed_gap` | the village is outside both NBN fixed-line and fixed-wireless footprints | nb 06 |
| `public_gap` | no RICT public phone / Wi-Fi site within 2 km | nb 06 |
| `school_gap` | nearest school > `SERVICE_GAP_KM` | this notebook |
| `medical_gap` | nearest medical facility > `SERVICE_GAP_KM` | this notebook |
| `emergency_gap` | nearest emergency facility > `SERVICE_GAP_KM` | this notebook |
| `no_connectivity` | `mobile_gap` **and** `fixed_gap` **and** `public_gap` -- no mobile, no fixed broadband, no public phone/Wi-Fi point (satellite only) | derived |
| `gap_score` | number of the six flags that are True (0-6) | derived |
| `gap_dimensions` | the True flags by name, `;`-joined (e.g. `mobile;fixed;school`) | derived |

No population is involved: population is an SA1 fact and can't be split across the villages inside
an SA1, so these flags describe villages, not people.


In [6]:
def as_bool(series):
    # village_gap.csv round-trip: a bool column can come back as object ("True"/"False") -- normalise
    return series.astype(str).str.strip().str.lower().eq("true")


v = village_gap_services_df
v["mobile_gap"] = as_bool(v["distance_gap_candidate"])
v["fixed_gap"] = ~(as_bool(v["in_nbn_fixed_line"]) | as_bool(v["in_nbn_fixed_wireless"]))
v["public_gap"] = ~as_bool(v["has_rict_public_access"])
v["school_gap"] = v["km_nearest_school"] > SERVICE_GAP_KM
v["medical_gap"] = v["km_nearest_medical"] > SERVICE_GAP_KM
v["emergency_gap"] = v["km_nearest_emergency"] > SERVICE_GAP_KM

GAP_FLAGS = ["mobile_gap", "fixed_gap", "public_gap", "school_gap", "medical_gap", "emergency_gap"]
v["no_connectivity"] = v["mobile_gap"] & v["fixed_gap"] & v["public_gap"]
v["gap_score"] = v[GAP_FLAGS].sum(axis=1).astype(int)
v["gap_dimensions"] = v[GAP_FLAGS].apply(
    lambda r: ";".join(f.removesuffix("_gap") for f in GAP_FLAGS if r[f]), axis=1)
GAP_COLUMNS = GAP_FLAGS + ["no_connectivity", "gap_score", "gap_dimensions"]


### Validate the gap flags


In [7]:
assert v["gap_score"].between(0, len(GAP_FLAGS)).all()
assert (v["gap_score"] == 0).eq(v["gap_dimensions"] == "").all()
assert (v["no_connectivity"] <= v["mobile_gap"]).all()

print("villages flagged, per dimension:")
print(v[GAP_FLAGS + ["no_connectivity"]].sum().to_string())
print("\ngap_score distribution (0 = no gap on any dimension):")
print(v["gap_score"].value_counts().sort_index().to_string())
print("\nno_connectivity x is_remote:")
print(pd.crosstab(v["no_connectivity"], v["is_remote"]))
print("\nmost common gap combinations:")
print(v["gap_dimensions"].replace("", "(none)").value_counts().head(8).to_string())


villages flagged, per dimension:
mobile_gap         384
fixed_gap          761
public_gap         518
school_gap         469
medical_gap        530
emergency_gap      556
no_connectivity    269

gap_score distribution (0 = no gap on any dimension):
gap_score
0     17
1     80
2    142
3     56
4     61
5    174
6    262

no_connectivity x is_remote:
is_remote        False  True 
no_connectivity              
False               13    510
True                 0    269

most common gap combinations:
gap_dimensions
mobile;fixed;public;school;medical;emergency    262
mobile;fixed;school;medical;emergency           103
fixed;public                                    101
fixed;public;school;medical;emergency            66
fixed                                            66
fixed;emergency                                  28
fixed;school;medical;emergency                   25
fixed;public;emergency                           23


## 5. Write outputs

Two files: the full `village_gap.csv` with the new distance columns and the section 4 gap flags appended (`data/processed/village_gap_with_services.csv`, which the Streamlit app reads; nb 06's own `village_gap.csv` is not touched), and a compact `community_id`-keyed table of just the new columns (`village_services_gap.csv`) for anyone who wants to join it onto something else.


In [8]:
out_csv = OUT_DIR / "village_gap_with_services.csv"
village_gap_services_df.to_csv(out_csv, index=False)
print(f"wrote {out_csv} ({village_gap_services_df.shape[0]} rows x {village_gap_services_df.shape[1]} cols)")

compact_cols = ["community_id", "community_name"] + list(results.keys()) + GAP_COLUMNS
compact_csv = OUT_DIR / "village_services_gap.csv"
village_gap_services_df[compact_cols].to_csv(compact_csv, index=False)
print(f"wrote {compact_csv}  ({len(village_gap_services_df)} rows x {len(compact_cols)} cols)")

wrote C:\Users\Aseem\Desktop\DIC\data\processed\village_gap_with_services.csv (792 rows x 52 cols)


wrote C:\Users\Aseem\Desktop\DIC\data\processed\village_services_gap.csv  (792 rows x 23 cols)


## 6. Known limitations (be upfront about these in the report)

- **School coordinates are mostly site precision, not all.** ACARA gives each school its own site; stand-alone preschools share their same-named school's point, and a few schools ACARA doesn't list fall back to a locality point. See `geocode_precision` in `services_sites.csv`.
- **Haversine, not EPSG:3577** -- unlike nb 06's tower/MBSP distances, these three columns are plain great-circle distance on WGS84. Close enough for triage; worth aligning with the rest of the pipeline's projected-CRS method if precision starts to matter.
- **`nearest_*_type` is a mixed bag per source** -- for `school` it's stage offered (Primary/Middle/...), for `medical` it's `NHSD_SERVICE_TYPE`, for `emergency` it's the facility class (Policing/SES/...). Don't compare this column across sources.
- **The gap cut-offs are triage choices, not standards.** 15 km (towers, services) and 2 km (RICT) are constants; change them in one place and re-run. `mobile_gap` inherits nb 06's guide-claim rule.
- **No population weighting** -- exactly like every other `km_*` column in this pipeline, per §2 of `PROJECT_CONTEXT.md`: population lives on `sa1_report.csv`, joined at SA1 level, never inferred from a village-level distance table.